In [1]:
import pandas as pd
import time
import numpy as np

In [2]:
# Complete Dataset
curated_bow = pd.read_csv('class_noise_cleaned.csv', sep=',')
curated_bow.dropna()
curated_bow.columns = [col.strip() for col in curated_bow.columns]
curated_bow

,id,line,class_value,path,testName,verdict,contents,class_name
0,0,0,1,0,0,0,"1,23 @@",build_status
1,0,0,1,0,0,0,This plugin downloads/installs Node and NPM lo...,build_status
2,0,0,1,0,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status
3,0,0,1,0,0,0,NaN,build_status
4,0,0,1,0,0,0,# Installing,build_status
...,...,...,...,...,...,...,...,...
1554,0,0,0,0,0,0,"12,7 @@ import java.util.List;",build_status
1555,0,0,0,0,0,0,factory.getJspmRunner().execute(argume...,build_status
1556,0,0,0,0,0,0,"public final void execute(String args, Map...",build_status
1557,0,0,0,0,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status


In [3]:
# Duplicate Columns Removed
curated_bow = curated_bow.loc[:,~curated_bow.columns.duplicated()].copy()
curated_bow

,id,line,class_value,path,testName,verdict,contents,class_name
0,0,0,1,0,0,0,"1,23 @@",build_status
1,0,0,1,0,0,0,This plugin downloads/installs Node and NPM lo...,build_status
2,0,0,1,0,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status
3,0,0,1,0,0,0,NaN,build_status
4,0,0,1,0,0,0,# Installing,build_status
...,...,...,...,...,...,...,...,...
1554,0,0,0,0,0,0,"12,7 @@ import java.util.List;",build_status
1555,0,0,0,0,0,0,factory.getJspmRunner().execute(argume...,build_status
1556,0,0,0,0,0,0,"public final void execute(String args, Map...",build_status
1557,0,0,0,0,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status


In [4]:
partitions_ = 1
bin_size = len(curated_bow.index)//partitions_ 
print(bin_size)

1559


In [5]:
def run_Panda():
    bins_counter = 0
    partition_values = pd.DataFrame()
    j_col= pd.DataFrame()
    k_col= pd.DataFrame()
    
    for j in range(len(curated_bow.columns)-2): 
        start_j = time.time()
        clustered_att = curated_bow.sort_values(by = curated_bow.columns[j])
        for k in range(len(curated_bow.columns)-2): 
            bins_counter = 0
            if k != j:
                for y in range(partitions_): 
                    
                    parition_mean = clustered_att.iloc[bins_counter: bins_counter + bin_size, j].values.mean()
                    partition_sd = clustered_att.iloc[bins_counter: bins_counter + bin_size, j].values.std()
                    mean_over_sd = parition_mean/partition_sd if partition_sd else 0

                    noise_score = pd.DataFrame([np.nan_to_num(abs(x - (mean_over_sd)))
                                                for x in clustered_att.iloc[bins_counter:bins_counter + bin_size, k]])

                    partition_values = pd.concat([partition_values, noise_score], axis=0)
                    
                    #dropping existing indices with the same name and replacing them with new, using reset index
                    partition_values= partition_values.reset_index(drop=True) 
                    bins_counter = bins_counter + bin_size
            else:
                continue
            
            k_col= pd.concat([k_col, partition_values], axis= 1)
            partition_values.drop(partition_values.index, inplace=True)
                
        
        j_col = pd.concat([j_col, k_col], axis = 1)
        end_j = time.time()
        print("column j: " + str(j))

        print(f"Runtime of the program for j {end_j - start_j}")
    return j_col

In [6]:
sss = run_Panda()
sss['max_noise'] = sss.max(axis=1)
# s = sss.sort_values(by='max_noise', ascending = False)

column j: 0
Runtime of the program for j 0.15149807929992676
column j: 1
Runtime of the program for j 0.15871858596801758
column j: 2
Runtime of the program for j 0.1500542163848877
column j: 3
Runtime of the program for j 0.1501772403717041
column j: 4
Runtime of the program for j 0.15489745140075684
column j: 5
Runtime of the program for j 0.15491127967834473


In [7]:
curated_bow['max_noise'] = sss['max_noise']
curated_bow

,id,line,class_value,path,testName,verdict,contents,class_name,max_noise
0,0,0,1,0,0,0,"1,23 @@",build_status,5.042195
1,0,0,1,0,0,0,This plugin downloads/installs Node and NPM lo...,build_status,5.042195
2,0,0,1,0,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status,5.042195
3,0,0,1,0,0,0,NaN,build_status,5.042195
4,0,0,1,0,0,0,# Installing,build_status,5.042195
...,...,...,...,...,...,...,...,...,...
1554,0,0,0,0,0,0,"12,7 @@ import java.util.List;",build_status,5.042195
1555,0,0,0,0,0,0,factory.getJspmRunner().execute(argume...,build_status,5.042195
1556,0,0,0,0,0,0,"public final void execute(String args, Map...",build_status,5.042195
1557,0,0,0,0,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status,5.042195


In [8]:
s = sss.sort_values(by='max_noise', ascending = False)
s = s.drop_duplicates(subset = 'max_noise')

values = []
for x in s['max_noise'].iloc[0:10]:
    values.append(x)

In [9]:
noise_sorted = curated_bow.sort_values(by='max_noise', ascending = False)
# df = df[df.rebounds.isin(values) == False]
noise_sorted.drop(noise_sorted[noise_sorted['max_noise'].isin(values) == True].index, axis = 0, inplace = True)
noise_sorted

,id,line,class_value,path,testName,verdict,contents,class_name,max_noise
